# ETFIQ MCP quickstart

Call the ETFIQ MCP server with plain `requests`, then load the answers into pandas. No key, no sign-in, read only.

Server: `https://mcp.etfiq.com/mcp`. It speaks JSON-RPC over HTTP, so any HTTP client works.
Every answer carries its source, its as-of date and a citation line. Free to use with credit and a link to ETFIQ.
Not investment advice.

In [1]:
import json
import requests
import pandas as pd

URL = "https://mcp.etfiq.com/mcp"
HEADERS = {"content-type": "application/json", "accept": "application/json, text/event-stream"}
_id = 0

def rpc(method, params=None):
    """One JSON-RPC call. Returns the result, or raises on a protocol error."""
    global _id
    _id += 1
    body = {"jsonrpc": "2.0", "id": _id, "method": method, "params": params or {}}
    r = requests.post(URL, headers=HEADERS, json=body, timeout=60)
    r.raise_for_status()
    text = r.text
    if text.startswith("event:"):  # server-sent event framing, if a server uses it
        text = next(l[5:] for l in text.splitlines() if l.startswith("data:"))
    msg = json.loads(text)
    if "error" in msg:
        raise RuntimeError(msg["error"])
    return msg["result"]

def call(tool, **arguments):
    """Call one tool. ETFIQ answers as {data, provenance, as_of, citation, ...} inside the text content."""
    res = rpc("tools/call", {"name": tool, "arguments": arguments})
    return json.loads(res["content"][0]["text"])

info = rpc("initialize", {"protocolVersion": "2025-03-26", "capabilities": {},
                          "clientInfo": {"name": "etfiq-quickstart-notebook", "version": "1"}})
tools = rpc("tools/list")["tools"]
print(info["serverInfo"]["name"], info["serverInfo"]["version"], "-", len(tools), "tools")
print(", ".join(t["name"] for t in tools))

etfiq 0.4.0 - 33 tools
etf_search, etf_profile, etf_holdings, etf_theme, etf_themes, etfiq_page, etfiq_sections, etfiq_list, etf_screen, etf_overlap, etfiq_as_of, etf_holders, etf_rankings, etf_flows, etfiq_methodology, find_page, list_benchmarks, get_benchmark, compare_to_benchmark, build_portfolio, etf_committee, etf_scorecard, etf_promise, etf_trading_cost, etf_return_of_capital, etf_benchmark, etf_compare, etf_movers, etf_news_today, strategy_list, strategy_signals, strategy_record, etf_risk_check


## 1. Fund flows into a DataFrame

`etf_flows` with no ticker ranks the biggest inflows and outflows over a window (`1D`, `1W`, `1M`, `3M`, `YTD`, `1Y`).

In [2]:
flows = call("etf_flows", window="1W", limit=10)
d = flows["data"]
print(f"{d['from']} to {d['to']}, {d['trading_days']} trading days, {d['counted']} funds counted")
print(f"Net flow across all funds: ${d['net_flow_usd']/1e9:,.1f}B")

cols = ["ticker", "name", "issuer", "category", "flow_usd", "through"]
df = pd.DataFrame(d["top_inflows"] + d["top_outflows"])[cols]
df["flow_usd_bn"] = (df["flow_usd"] / 1e9).round(2)
df.drop(columns="flow_usd")

2026-10-29 to 2026-11-04, 5 trading days, 2248 funds counted
Net flow across all funds: $37.9B


,ticker,name,issuer,category,through,flow_usd_bn
0,SPY,SPDR S&P 500 ETF Trust,State Street,Stocks,2026-11-04,9.38
1,QQQ,"Invesco QQQ Trust, Series 1",Invesco,Stocks,2026-11-04,6.11
2,IVV,iShares Core S&P 500 ETF,iShares,Stocks,2026-11-04,4.05
3,SOXL,Direxion Daily Semiconductor Bull 3X ETF,Direxion,NaN,2026-11-04,3.40
4,RSP,Invesco S&P 500 Equal Weight ETF,Invesco,Stocks,2026-11-04,1.57
5,LQD,iShares iBoxx $ Investment Grade Corporate Bon...,iShares,Bonds,2026-11-04,1.48
6,IEF,iShares 7-10 Year Treasury Bond ETF,iShares,Bonds,2026-11-04,1.43
7,TLT,iShares 20+ Year Treasury Bond ETF,iShares,Bonds,2026-11-04,1.29
8,QQQM,Invesco NASDAQ 100 ETF,Invesco,Stocks,2026-11-04,1.13
9,TQQQ,ProShares UltraPro QQQ,ProShares,NaN,2026-11-04,1.00


In [3]:
print(flows.get("citation"))
print(json.dumps(flows.get("provenance"), indent=1)[:600])

ETFIQ, as of 2026-11-04. https://etfiq.com/flows/
[
 {
  "field": "flow_usd",
  "source": "ETFIQ, from each fund's daily change in shares outstanding times its NAV",
  "as_of": "2026-11-04",
  "source_uri": "https://etfiq.com/methodology/flows"
 }
]


## 2. Holdings table

`etf_holdings` returns the issuer's daily file, weights in percent of net assets. The record says which file and which date.

In [4]:
h = call("etf_holdings", ticker="SPY", top_n=10)
holdings = pd.DataFrame(h["data"]["holdings"])
print(f"{h['data']['ticker']}: {h['data']['total_holdings']} holdings, source: {h['data']['source_kind']}, as of {h['as_of']}")
print(h["citation"])
holdings

SPY: 504 holdings, source: issuer daily file, as of 2026-11-04
ETFIQ, as of 2026-11-04. https://etfiq.com/funds/spy


,name,ticker,weight_pct
0,APPLE INC,AAPL,7.690
1,NVIDIA CORP,NVDA,7.302
2,MICROSOFT CORP,MSFT,6.222
3,AMAZON.COM INC,AMZN,3.937
4,ALPHABET INC CL A,GOOGL,2.556
5,BROADCOM INC,AVGO,2.334
6,META PLATFORMS INC CLASS A,META,2.094
7,ALPHABET INC CL C,GOOG,2.059
8,ELI LILLY + CO,LLY,1.710
9,TESLA INC,TSLA,1.637


## 3. Point in time: `etfiq_as_of`

`etfiq_as_of` returns what ETFIQ published on a past date. The research pieces from that day are frozen at that date, so the same question gives the same answer later.

In [5]:
past = call("etfiq_as_of", date="2026-10-01")
d = past["data"]
print(f"Published on {d['date']}: {d['pieces_published']} research pieces. Page: {d['url']}")
pieces = pd.DataFrame(d["pieces"])[["slug", "url"]]
print({k: past[k] for k in ("as_of", "citation") if k in past})
pieces

Published on 2026-10-01: 6 research pieces. Page: https://etfiq.com/research/2026-10-01/
{'as_of': '2026-10-01', 'citation': 'ETFIQ, as of 2026-10-01. https://etfiq.com/research/2026-10-01/'}


,slug,url
0,alternatives-down-weeks,https://etfiq.com/research/2026-10-01/alternat...
1,buffer-state,https://etfiq.com/research/2026-10-01/buffer-s...
2,crypto-tracking,https://etfiq.com/research/2026-10-01/crypto-t...
3,income-ahead,https://etfiq.com/research/2026-10-01/income-a...
4,leverage-label,https://etfiq.com/research/2026-10-01/leverage...
5,themes-index,https://etfiq.com/research/2026-10-01/themes-i...


## Data provenance and citation

- Every tool answer has `provenance` (field, source, as-of date, source URI), `as_of` and a ready `citation` string. Use it.
- Credit as: **Source: ETFIQ (etfiq.com), data as of <date>**, with a link to the page the answer names.
- ETFIQ's computed numbers are free to use with credit and a link to ETFIQ. Numbers an issuer publishes, such as holdings files, stay under the issuer's terms and are credited to it in `provenance`.
- A tool that cannot answer refuses and says why. It does not return a near match.

Licence: https://etfiq.com/license/